# ROS Bag `.db3` Inspection

This notebook gives a screenshot-friendly view of one ROS 2 bag database. It shows the bag folder, `metadata.yaml` summary, SQLite tables, recorded topics, and timestamped message rows.

The `.db3` file stores serialized ROS messages, so this notebook shows the database structure and message metadata rather than trying to print raw binary message content.

In [1]:
from pathlib import Path
from datetime import datetime
import re
import sqlite3
import yaml

try:
    import pandas as pd
except ImportError:
    pd = None

THESIS_ROOT = Path.home() / "Documents" / "Thesis"
BAG_DIR = THESIS_ROOT / "dataset" / "bags" / "run_model_20260522_201207"
METADATA_PATH = BAG_DIR / "metadata.yaml"
DB3_PATH = next(BAG_DIR.glob("*.db3"))

print("Bag directory:", BAG_DIR)
print("Metadata:", METADATA_PATH)
print("DB3:", DB3_PATH)

Bag directory: /home/basudeo/Documents/Thesis/dataset/bags/run_model_20260522_201207
Metadata: /home/basudeo/Documents/Thesis/dataset/bags/run_model_20260522_201207/metadata.yaml
DB3: /home/basudeo/Documents/Thesis/dataset/bags/run_model_20260522_201207/run_model_20260522_201207_0.db3


## 1. Bag Summary From `metadata.yaml`

In [2]:
metadata = yaml.safe_load(METADATA_PATH.read_text())
info = metadata["rosbag2_bagfile_information"]

duration_ns = int(info["duration"]["nanoseconds"])
summary_rows = [
    {"Item": "Storage format", "Value": info["storage_identifier"]},
    {"Item": "Duration (seconds)", "Value": round(duration_ns / 1e9, 3)},
    {"Item": "Total message count", "Value": int(info["message_count"])},
    {"Item": "Starting time (ns)", "Value": int(info["starting_time"]["nanoseconds_since_epoch"])},
    {"Item": "Recorded topics", "Value": len(info["topics_with_message_count"])},
]

if pd:
    display(pd.DataFrame(summary_rows))
else:
    for row in summary_rows:
        print(row)

,Item,Value
0,Storage format,sqlite3
1,Duration (seconds),310.25
2,Total message count,167086
3,Starting time (ns),1779473528096638017
4,Recorded topics,36


## 2. Recorded Topics And Message Counts

In [3]:
topic_rows = []
for entry in info["topics_with_message_count"]:
    topic = entry["topic_metadata"]
    topic_rows.append({
        "Topic": topic["name"],
        "Message type": topic["type"],
        "Message count": int(entry["message_count"]),
    })

if pd:
    topics_df = pd.DataFrame(topic_rows).sort_values("Message count", ascending=False)
    display(topics_df.head(20))
else:
    for row in sorted(topic_rows, key=lambda item: item["Message count"], reverse=True)[:20]:
        print(row)

,Topic,Message type,Message count
26,/clock,rosgraph_msgs/msg/Clock,108193
0,/world/baylands/dynamic_pose/info,tf2_msgs/msg/TFMessage,17312
27,/model/uav1/odometry,nav_msgs/msg/Odometry,4925
31,/model/uav2/odometry,nav_msgs/msg/Odometry,4925
8,/model/husky_2/odometry,nav_msgs/msg/Odometry,4925
7,/uav1/controller_state,std_msgs/msg/String,2078
12,/model/uav1/command/twist,geometry_msgs/msg/Twist,2077
33,/uav1/command/twist,geometry_msgs/msg/Twist,2077
3,/uav2/controller_state,std_msgs/msg/String,2041
29,/model/uav2/command/twist,geometry_msgs/msg/Twist,2040


## 3. SQLite Tables In The `.db3` File

In [4]:
conn = sqlite3.connect(DB3_PATH)
tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()
table_rows = [{"Table": row[0]} for row in tables]

if pd:
    display(pd.DataFrame(table_rows))
else:
    for row in table_rows:
        print(row)

,Table
0,messages
1,metadata
2,schema
3,topics


## 4. Important Database Schemas

In [5]:
def table_schema(table_name):
    rows = conn.execute(f"PRAGMA table_info({table_name})").fetchall()
    return [
        {"Column id": row[0], "Name": row[1], "Type": row[2], "Required": bool(row[3]), "Primary key": bool(row[5])}
        for row in rows
    ]

topics_schema = table_schema("topics")
messages_schema = table_schema("messages")

if pd:
    print("topics table")
    display(pd.DataFrame(topics_schema))
    print("messages table")
    display(pd.DataFrame(messages_schema))
else:
    print("topics table", topics_schema)
    print("messages table", messages_schema)

topics table


,Column id,Name,Type,Required,Primary key
0,0,id,INTEGER,False,True
1,1,name,TEXT,True,False
2,2,type,TEXT,True,False
3,3,serialization_format,TEXT,True,False
4,4,offered_qos_profiles,TEXT,True,False


messages table


,Column id,Name,Type,Required,Primary key
0,0,id,INTEGER,False,True
1,1,topic_id,INTEGER,True,False
2,2,timestamp,INTEGER,True,False
3,3,data,BLOB,True,False


## 5. Example Timestamped Message Rows

Each row below is one recorded ROS message. The `data` field is stored as a serialized binary blob, so this table shows the timestamp, topic, message type, and binary size.

In [6]:
bag_start_ns = conn.execute("SELECT MIN(timestamp) FROM messages").fetchone()[0]

query = """
SELECT
    messages.id AS message_id,
    messages.timestamp AS timestamp_ns,
    ROUND((messages.timestamp - ?) / 1000000000.0, 6) AS time_from_start_s,
    topics.name AS topic,
    topics.type AS message_type,
    LENGTH(messages.data) AS serialized_data_bytes
FROM messages
JOIN topics ON messages.topic_id = topics.id
WHERE topics.name != '/clock'
ORDER BY messages.timestamp
LIMIT 12
"""

rows = conn.execute(query, (bag_start_ns,)).fetchall()
columns = ["message_id", "timestamp_ns", "time_from_start_s", "topic", "message_type", "serialized_data_bytes"]
rows = [
    (*row[:3], datetime.fromtimestamp(row[1] / 1e9).strftime("%Y-%m-%d %H:%M:%S.%f")[:-3], *row[3:])
    for row in rows
]
columns = ["message_id", "timestamp_ns", "time_from_start_s", "datetime", "topic", "message_type", "serialized_data_bytes"]

if pd:
    display(pd.DataFrame(rows, columns=columns))
else:
    for row in rows:
        print(dict(zip(columns, row)))

,message_id,timestamp_ns,time_from_start_s,datetime,topic,message_type,serialized_data_bytes
0,2,1779473528096653678,0.000016,2026-05-22 20:12:08.096,/world/baylands/dynamic_pose/info,tf2_msgs/msg/TFMessage,1940
1,4,1779473528096700812,0.000063,2026-05-22 20:12:08.096,/world/baylands/dynamic_pose/info,tf2_msgs/msg/TFMessage,1940
2,22,1779473528113815137,0.017177,2026-05-22 20:12:08.113,/world/baylands/dynamic_pose/info,tf2_msgs/msg/TFMessage,1940
3,23,1779473528113850521,0.017213,2026-05-22 20:12:08.113,/world/baylands/dynamic_pose/info,tf2_msgs/msg/TFMessage,1940
4,38,1779473528131669757,0.035032,2026-05-22 20:12:08.131,/world/baylands/dynamic_pose/info,tf2_msgs/msg/TFMessage,1940
5,39,1779473528131728748,0.035091,2026-05-22 20:12:08.131,/world/baylands/dynamic_pose/info,tf2_msgs/msg/TFMessage,1940
6,54,1779473528149186573,0.052549,2026-05-22 20:12:08.149,/world/baylands/dynamic_pose/info,tf2_msgs/msg/TFMessage,1940
7,55,1779473528149241871,0.052604,2026-05-22 20:12:08.149,/world/baylands/dynamic_pose/info,tf2_msgs/msg/TFMessage,1940
8,69,1779473528165842599,0.069205,2026-05-22 20:12:08.165,/world/baylands/dynamic_pose/info,tf2_msgs/msg/TFMessage,1940
9,70,1779473528165875135,0.069237,2026-05-22 20:12:08.165,/world/baylands/dynamic_pose/info,tf2_msgs/msg/TFMessage,1940


## 6. Example Rows For Training-Relevant Topics

In [7]:
topic_filter = [
    "/model/husky_2/odometry",
    "/cmd_vel_husky2",
    "/husky_2/controller_state",
    "/husky_2/obstacle_action",
    "/husky_2/obstacle_clearance",
    "/episode/husky_2/start",
    "/episode/husky_2/goal",
    "/model/uav1/odometry",
    "/model/uav2/odometry",
]

placeholders = ",".join("?" for _ in topic_filter)
query = f"""
SELECT
    messages.id AS message_id,
    messages.timestamp AS timestamp_ns,
    ROUND((messages.timestamp - ?) / 1000000000.0, 6) AS time_from_start_s,
    topics.name AS topic,
    topics.type AS message_type,
    LENGTH(messages.data) AS serialized_data_bytes
FROM messages
JOIN topics ON messages.topic_id = topics.id
WHERE topics.name IN ({placeholders})
ORDER BY messages.timestamp
LIMIT 20
"""
rows = conn.execute(query, (bag_start_ns, *topic_filter)).fetchall()
rows = [
    (*row[:3], datetime.fromtimestamp(row[1] / 1e9).strftime("%Y-%m-%d %H:%M:%S.%f")[:-3], *row[3:])
    for row in rows
]
columns = ["message_id", "timestamp_ns", "time_from_start_s", "datetime", "topic", "message_type", "serialized_data_bytes"]

if pd:
    display(pd.DataFrame(rows, columns=columns))
else:
    for row in rows:
        print(dict(zip(columns, row)))

,message_id,timestamp_ns,time_from_start_s,datetime,topic,message_type,serialized_data_bytes
0,4263,1779473532778361135,4.681723,2026-05-22 20:12:12.778,/episode/husky_2/start,geometry_msgs/msg/PoseStamped,84
1,4264,1779473532778407720,4.681770,2026-05-22 20:12:12.778,/episode/husky_2/goal,geometry_msgs/msg/PoseStamped,84
2,4276,1779473532784240323,4.687602,2026-05-22 20:12:12.784,/husky_2/controller_state,std_msgs/msg/String,20
3,4321,1779473532834670269,4.738032,2026-05-22 20:12:12.834,/husky_2/controller_state,std_msgs/msg/String,20
4,4333,1779473532845991914,4.749354,2026-05-22 20:12:12.845,/cmd_vel_husky2,geometry_msgs/msg/Twist,52
5,4418,1779473532934776904,4.838139,2026-05-22 20:12:12.934,/husky_2/controller_state,std_msgs/msg/String,16
6,4419,1779473532934880350,4.838242,2026-05-22 20:12:12.934,/cmd_vel_husky2,geometry_msgs/msg/Twist,52
7,4528,1779473533046896024,4.950258,2026-05-22 20:12:13.046,/husky_2/controller_state,std_msgs/msg/String,16
8,4529,1779473533047005053,4.950367,2026-05-22 20:12:13.047,/cmd_vel_husky2,geometry_msgs/msg/Twist,52
9,4615,1779473533136116583,5.039479,2026-05-22 20:12:13.136,/husky_2/controller_state,std_msgs/msg/String,16


## 7. Topic Frequencies Computed Directly From `.db3`

This table uses only the SQLite database. For each topic, it counts messages and uses the first and last message timestamps in the `messages` table.

Formula:

`frequency_hz = message_count / duration_seconds`

`interval_seconds = 1 / frequency_hz`

In [8]:
frequency_query = """
SELECT
    topics.name AS topic,
    topics.type AS message_type,
    COUNT(messages.id) AS message_count,
    MIN(messages.timestamp) AS first_timestamp_ns,
    MAX(messages.timestamp) AS last_timestamp_ns
FROM messages
JOIN topics ON messages.topic_id = topics.id
GROUP BY topics.id, topics.name, topics.type
ORDER BY message_count DESC
"""

frequency_rows = []
for topic, message_type, count, first_ns, last_ns in conn.execute(frequency_query):
    duration_s = (int(last_ns) - int(first_ns)) / 1e9 if count > 1 else 0.0
    frequency_hz = (int(count) / duration_s) if duration_s > 0 else 0.0
    interval_s = (1.0 / frequency_hz) if frequency_hz > 0 else 0.0
    frequency_rows.append({
        "Topic": topic,
        "Message type": message_type,
        "Message count": int(count),
        "Start datetime": datetime.fromtimestamp(int(first_ns) / 1e9).strftime("%Y-%m-%d %H:%M:%S.%f")[:-3],
        "End datetime": datetime.fromtimestamp(int(last_ns) / 1e9).strftime("%Y-%m-%d %H:%M:%S.%f")[:-3],
        "Duration s": round(duration_s, 3),
        "Frequency Hz": round(frequency_hz, 3),
        "Interval s": round(interval_s, 4),
    })

if pd:
    frequency_df = pd.DataFrame(frequency_rows)
    display(frequency_df)
else:
    for row in frequency_rows:
        print(row)

,Topic,Message type,Message count,Start datetime,End datetime,Duration s,Frequency Hz,Interval s
0,/clock,rosgraph_msgs/msg/Clock,108193,2026-05-22 20:12:08.096,2026-05-22 20:17:16.799,308.703,350.476,0.0029
1,/world/baylands/dynamic_pose/info,tf2_msgs/msg/TFMessage,17312,2026-05-22 20:12:08.096,2026-05-22 20:17:16.792,308.696,56.081,0.0178
2,/model/husky_2/odometry,nav_msgs/msg/Odometry,4925,2026-05-22 20:12:19.681,2026-05-22 20:17:13.320,293.639,16.772,0.0596
3,/model/uav1/odometry,nav_msgs/msg/Odometry,4925,2026-05-22 20:12:19.681,2026-05-22 20:17:13.319,293.639,16.772,0.0596
4,/model/uav2/odometry,nav_msgs/msg/Odometry,4925,2026-05-22 20:12:19.681,2026-05-22 20:17:13.320,293.639,16.772,0.0596
5,/uav1/controller_state,std_msgs/msg/String,2078,2026-05-22 20:12:12.778,2026-05-22 20:17:18.346,305.568,6.800,0.1470
6,/model/uav1/command/twist,geometry_msgs/msg/Twist,2077,2026-05-22 20:12:12.846,2026-05-22 20:17:18.346,305.500,6.799,0.1471
7,/uav1/command/twist,geometry_msgs/msg/Twist,2077,2026-05-22 20:12:12.846,2026-05-22 20:17:18.346,305.500,6.799,0.1471
8,/uav2/controller_state,std_msgs/msg/String,2041,2026-05-22 20:12:12.780,2026-05-22 20:17:18.258,305.478,6.681,0.1497
9,/model/uav2/command/twist,geometry_msgs/msg/Twist,2040,2026-05-22 20:12:12.861,2026-05-22 20:17:18.258,305.397,6.680,0.1497


## 8. Screenshot-Friendly Frequency Summary

This shorter table is easier to place in slides.

In [9]:
important_topics = [
    "/world/baylands/dynamic_pose/info",
    "/model/husky_2/odometry",
    "/cmd_vel_husky2",
    "/husky_2/controller_state",
    "/husky_2/obstacle_action",
    "/husky_2/obstacle_clearance",
    "/world/baylands/model/husky_2/link/base_link/sensor/planar_laser/scan",
    "/world/baylands/model/husky_2/link/base_link/sensor/front_laser/scan/points",
    "/model/uav1/odometry",
    "/model/uav2/odometry",
    "/episode/husky_2/start",
    "/episode/husky_2/goal",
]

summary_rows = [row for row in frequency_rows if row["Topic"] in important_topics]
if pd:
    summary_df = pd.DataFrame(summary_rows)[[
        "Topic", "Message type", "Message count", "Start datetime", "End datetime", "Duration s", "Frequency Hz", "Interval s"
    ]]
    display(summary_df)
else:
    for row in summary_rows:
        print({key: row[key] for key in ["Topic", "Message type", "Message count", "Start datetime", "End datetime", "Duration s", "Frequency Hz", "Interval s"]})

,Topic,Message type,Message count,Start datetime,End datetime,Duration s,Frequency Hz,Interval s
0,/world/baylands/dynamic_pose/info,tf2_msgs/msg/TFMessage,17312,2026-05-22 20:12:08.096,2026-05-22 20:17:16.792,308.696,56.081,0.0178
1,/model/husky_2/odometry,nav_msgs/msg/Odometry,4925,2026-05-22 20:12:19.681,2026-05-22 20:17:13.320,293.639,16.772,0.0596
2,/model/uav1/odometry,nav_msgs/msg/Odometry,4925,2026-05-22 20:12:19.681,2026-05-22 20:17:13.319,293.639,16.772,0.0596
3,/model/uav2/odometry,nav_msgs/msg/Odometry,4925,2026-05-22 20:12:19.681,2026-05-22 20:17:13.320,293.639,16.772,0.0596
4,/husky_2/controller_state,std_msgs/msg/String,1997,2026-05-22 20:12:12.784,2026-05-22 20:17:18.330,305.546,6.536,0.1530
5,/cmd_vel_husky2,geometry_msgs/msg/Twist,1996,2026-05-22 20:12:12.845,2026-05-22 20:17:18.330,305.484,6.534,0.1530
6,/husky_2/obstacle_clearance,geometry_msgs/msg/Vector3,986,2026-05-22 20:12:20.931,2026-05-22 20:17:13.572,292.641,3.369,0.2968
7,/husky_2/obstacle_action,std_msgs/msg/String,986,2026-05-22 20:12:20.931,2026-05-22 20:17:13.572,292.641,3.369,0.2968
8,/world/baylands/model/husky_2/link/base_link/s...,sensor_msgs/msg/PointCloud2,986,2026-05-22 20:12:20.965,2026-05-22 20:17:13.398,292.433,3.372,0.2966
9,/world/baylands/model/husky_2/link/base_link/s...,sensor_msgs/msg/LaserScan,986,2026-05-22 20:12:20.927,2026-05-22 20:17:13.372,292.444,3.372,0.2966


## 9. Unique Topic Groups Without Robot Names

This table removes robot-specific names such as `uav1`, `uav2`, `husky_2`, and `husky_local`. It gives one compact row per unique topic group and message type, which is easier to use in slides.

In [10]:
def normalize_topic_name(topic: str) -> str:
    normalized = topic
    normalized = re.sub(r"/world/([^/]+)/model/(uav\d+|husky_\d+|husky_local)/", r"/world/\1/model/<robot>/", normalized)
    normalized = re.sub(r"/model/(uav\d+|husky_\d+|husky_local)/", "/model/<robot>/", normalized)
    normalized = re.sub(r"/episode/(uav\d+|husky_\d+|husky_local)/", "/episode/<robot>/", normalized)
    normalized = re.sub(r"/(uav\d+|husky_\d+|husky_local)/", "/<robot>/", normalized)
    normalized = re.sub(r"^/cmd_vel(_husky\d+)?$", "/cmd_vel_<robot>", normalized)
    return normalized

grouped = {}
for row in frequency_rows:
    topic_group = normalize_topic_name(row["Topic"])
    key = (topic_group, row["Message type"])
    grouped.setdefault(key, []).append(row)

unique_rows = []
for (topic_group, message_type), items in sorted(grouped.items(), key=lambda item: item[0][0]):
    frequencies = [item["Frequency Hz"] for item in items]
    intervals = [item["Interval s"] for item in items]
    counts = [item["Message count"] for item in items]
    unique_rows.append({
        "Topic group": topic_group,
        "Message type": message_type,
        "Topics grouped": len(items),
        "Example original topic": items[0]["Topic"],
        "Message count range": f"{min(counts)}-{max(counts)}" if min(counts) != max(counts) else str(counts[0]),
        "Frequency Hz range": f"{min(frequencies):.3f}-{max(frequencies):.3f}" if min(frequencies) != max(frequencies) else f"{frequencies[0]:.3f}",
        "Interval s range": f"{min(intervals):.4f}-{max(intervals):.4f}" if min(intervals) != max(intervals) else f"{intervals[0]:.4f}",
    })

if pd:
    unique_df = pd.DataFrame(unique_rows)
    display(unique_df)
else:
    for row in unique_rows:
        print(row)


,Topic group,Message type,Topics grouped,Example original topic,Message count range,Frequency Hz range,Interval s range
0,/<robot>/command/twist,geometry_msgs/msg/Twist,2,/uav1/command/twist,2040-2077,6.680-6.799,0.1471-0.1497
1,/<robot>/controller_state,std_msgs/msg/String,3,/uav1/controller_state,1997-2078,6.536-6.800,0.1470-0.1530
2,/<robot>/obstacle_action,std_msgs/msg/String,3,/husky_2/obstacle_action,648-986,2.226-3.369,0.2968-0.4492
3,/<robot>/obstacle_clearance,geometry_msgs/msg/Vector3,3,/husky_2/obstacle_clearance,648-986,2.226-3.369,0.2968-0.4492
4,/clock,rosgraph_msgs/msg/Clock,1,/clock,108193,350.476,0.0029
5,/cmd_vel_<robot>,geometry_msgs/msg/Twist,1,/cmd_vel_husky2,1996,6.534,0.1530
6,/episode/<robot>/goal,geometry_msgs/msg/PoseStamped,3,/episode/husky_2/goal,306,1.004,0.9964
7,/episode/<robot>/start,geometry_msgs/msg/PoseStamped,3,/episode/husky_2/start,306,1.004,0.9964
8,/model/<robot>/command/twist,geometry_msgs/msg/Twist,2,/model/uav1/command/twist,2040-2077,6.680-6.799,0.1471-0.1497
9,/model/<robot>/odometry,nav_msgs/msg/Odometry,3,/model/husky_2/odometry,4925,16.772,0.0596


## 10. Slide Explanation

Use this wording in the presentation:

> The ROS bag folder contains a readable `metadata.yaml` file and a `.db3` SQLite database. The metadata file lists the recorded topics, message types, message counts, and duration. The `.db3` file stores each recorded message as a timestamp, topic id, and serialized binary data. During export, these serialized messages are deserialized, read in timestamp order, and converted into synchronized episode frames for model training.